In [30]:
import copy
from pathlib import Path
from typing import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
from natsort import natsorted
import pymeshlab

from plant_shape_analysis.dataloaders.trackplant3D import LeafSequencesDataset
from plant_shape_analysis.data_preprocessing.leaf_meshing import mesh_leaf

from plant_shape_analysis.reconstruction import classical_methods

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Mesh example single leaf

In [31]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [32]:
leaf_dataset.get_sequence_names()

['maize_control_plant1_leaf1',
 'maize_control_plant1_leaf2',
 'maize_control_plant1_leaf3',
 'maize_control_plant1_leaf4',
 'maize_control_plant3_leaf1',
 'maize_control_plant3_leaf2',
 'maize_control_plant3_leaf3',
 'maize_control_plant3_leaf4',
 'maize_control_plant2_leaf1',
 'maize_control_plant2_leaf2',
 'maize_control_plant2_leaf3',
 'maize_control_plant2_leaf4',
 'maize_control_plant2_leaf5',
 'sorghum_shade_plant2_leaf1',
 'sorghum_shade_plant2_leaf2',
 'sorghum_shade_plant2_leaf3',
 'sorghum_heat_plant1_leaf1',
 'sorghum_heat_plant1_leaf2',
 'sorghum_heat_plant1_leaf3',
 'sorghum_heat_plant1_leaf4',
 'sorghum_shade_plant3_leaf1',
 'sorghum_shade_plant3_leaf2',
 'sorghum_shade_plant3_leaf3',
 'sorghum_highlight_plant2_leaf1',
 'sorghum_highlight_plant2_leaf2',
 'sorghum_highlight_plant2_leaf3',
 'sorghum_highlight_plant2_leaf4',
 'sorghum_highlight_plant2_leaf5',
 'sorghum_highlight_plant2_leaf6',
 'sorghum_shade_plant1_leaf1',
 'sorghum_shade_plant1_leaf2',
 'sorghum_shade_pla

In [33]:
maize_leaves = leaf_dataset.get_timeseries_by_plant_sequence('maize_control_plant1')
maize_leaves

[{'sequence_name': 'maize_control_plant1_leaf1',
  'plant_sequence_name': 'maize_control_plant1',
  'leaf_id': 1,
  'timepoints': [{'day': 0,
    'leaf_id': 1,
    'points': array([[ 6.528425, -4.4796  ,  3.937625],
           [ 6.42105 , -8.4412  , 20.428499],
           [-2.45315 , -4.028133, 12.709   ],
           ...,
           [ 7.125534, -6.243192,  7.31445 ],
           [-0.10225 , -2.6813  ,  3.67935 ],
           [-5.865   ,  1.13345 , -2.873725]], shape=(5594, 3)),
    'dense_points': array([[ -4.184584,   6.560559, -12.768311],
           [ -4.346184,   6.494059, -12.859811],
           [ -4.381384,   6.433059, -12.925611],
           ...,
           [ -5.713584,   3.374759, -13.906611],
           [ -5.626284,   3.374859, -13.941411],
           [ -5.537884,   3.372259, -13.974911]], shape=(155504, 3)),
    'leaf_tip': array([ 7.34715 , -9.7354  , 19.847799]),
    'file_path': PosixPath('../data/TrackPlant3D/gt_corrected_v1/maize/1_maize_control_plant1_D00.txt')},
   {'day

In [34]:
leaf_dataset.get_timeseries_by_sequence_name('maize_control_plant1_leaf1')

{'sequence_name': 'maize_control_plant1_leaf1',
 'plant_sequence_name': 'maize_control_plant1',
 'leaf_id': 1,
 'timepoints': [{'day': 0,
   'leaf_id': 1,
   'points': array([[ 6.528425, -4.4796  ,  3.937625],
          [ 6.42105 , -8.4412  , 20.428499],
          [-2.45315 , -4.028133, 12.709   ],
          ...,
          [ 7.125534, -6.243192,  7.31445 ],
          [-0.10225 , -2.6813  ,  3.67935 ],
          [-5.865   ,  1.13345 , -2.873725]], shape=(5594, 3)),
   'dense_points': array([[ -4.184584,   6.560559, -12.768311],
          [ -4.346184,   6.494059, -12.859811],
          [ -4.381384,   6.433059, -12.925611],
          ...,
          [ -5.713584,   3.374759, -13.906611],
          [ -5.626284,   3.374859, -13.941411],
          [ -5.537884,   3.372259, -13.974911]], shape=(155504, 3)),
   'leaf_tip': array([ 7.34715 , -9.7354  , 19.847799]),
   'file_path': PosixPath('../data/TrackPlant3D/gt_corrected_v1/maize/1_maize_control_plant1_D00.txt')},
  {'day': 1,
   'leaf_id': 1,

In [ ]:
seq_id = 0
timepoint = 0
leaf = maize_leaves[seq_id]['timepoints'][timepoint]
leaf

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="pymeshlab")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="open3d")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="pymeshlab")
mesh.get_surface_area()

In [ ]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="open3d")
mesh.get_surface_area()

In [ ]:
o3d.visualization.draw_geometries(
[mesh],
point_show_normal=True,
window_name="Leaf Point Cloud and normals",
)

## Extract all leaves (dense and sparse)

We save without any temporal alignment, so then we can do temporal alignment later if needed and apply the same transformation to all sparse and dense point clouds and mesh simultaneously.

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
    apply_pca_alignment=True,
    save_transformations=True, # TODO: avoid recomputing transformations if they exist
)

Dense leaves

In [ ]:
for seq in leaf_dataset:
    for scan in seq['timepoints']:
        if scan['dense_points'] is not None:
            # save leaf
            crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
            file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}_dense.ply")
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(scan['dense_points'])
            save_dir = Path(f"../data/TrackPlant3D/dense_leaves_aligned/{crop}")
            save_dir.mkdir(parents=True, exist_ok=True)
            o3d.io.write_point_cloud(save_dir / file_name, pcd)

Also extract sparse leaves

In [ ]:
for seq in leaf_dataset:
    for scan in seq['timepoints']:
        # save leaf
        crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
        file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}.ply")
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(scan['points'])
        save_dir = Path(f"../data/TrackPlant3D/sparse_leaves_aligned/{crop}")
        save_dir.mkdir(parents=True, exist_ok=True)
        o3d.io.write_point_cloud(save_dir / file_name, pcd)

CSV file with all leaves to do quality assessment

In [ ]:
# import re
# import pandas as pd

# # Get file names
# dataset_path = Path("../data/TrackPlant3D/dense_leaves")
# files = []
# for crop in ['maize', 'tomato']:
#     crop = [f.stem for f in (dataset_path / crop).iterdir()]
#     files.extend(crop)
# files = natsorted(files)

# # Helper func
# def extract_numeric_suffix(x):
#     return int(re.search(r"\d+$", x).group(0))

# df = pd.DataFrame(files, columns=['file_name'])
# df['crop'] = df['file_name'].apply(lambda x: x.split('_')[1])
# df['treatment'] = df['file_name'].apply(lambda x: x.split('_')[2])
# df['plant'] = df['file_name'].apply(lambda x: extract_numeric_suffix(x.split('_')[3]))
# df['day'] = df['file_name'].apply(lambda x: extract_numeric_suffix(x.split('_')[4]))
# df['leaf'] = df['file_name'].apply(lambda x: extract_numeric_suffix(x.split('_')[5]))
# df['leaf_sequence_id'] = df.groupby(['crop', 'treatment','plant', 'leaf']).ngroup()
# df.insert(0, 'leaf_sequence_id', df.pop('leaf_sequence_id'))
# df['dense_leaf_id'] = df.index
# df.insert(0, 'dense_leaf_id', df.pop('dense_leaf_id'))
# df.to_csv("../data/TrackPlant3D/dense_leaves/dense_leaves_quality_assesment.csv", index=False, delimiter=';')

## Quality assurance and outlier removal

In [ ]:
crop = 'tomato' # 'maize'
idx = 217
leaf_paths = natsorted(Path(f"../data/TrackPlant3D/dense_leaves/{crop}").iterdir())
if crop == 'tomato':
    # Adjust index because we are skipping maize files
    maize_files = natsorted(Path("../data/TrackPlant3D/dense_leaves/maize").iterdir())
    idx -= len(maize_files)
leaf_paths[idx]

In [ ]:
pcd = o3d.t.io.read_point_cloud(str(leaf_paths[idx]))
print(f"Loaded point cloud with {len(pcd.to_legacy().points)} points")
pcd.estimate_normals()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!
pcd.orient_normals_to_align_with_direction() # This seems even more important 
# o3d.visualization.draw_geometries([pcd.to_legacy()], point_show_normal=True, window_name="Original point cloud")

Outlier removal

In [ ]:
# statistical outlier removal
clean_pcd, clean_idx = pcd.to_legacy().remove_statistical_outlier(nb_neighbors=50, std_ratio=5)
removed_indices = np.setdiff1d(np.arange(len(pcd.to_legacy().points)), np.asarray(clean_idx))
print(f"Removed {len(removed_indices)} outlier points.")
#o3d.visualization.draw_geometries([clean_pcd], window_name="Cleaned point cloud")

In [ ]:
def display_inlier_outlier(cloud, ind, visualize_separately=False):
    inlier_cloud = cloud.select_by_index(ind)
    outlier_cloud = cloud.select_by_index(ind, invert=True)

    print("Showing outliers (red) and inliers (gray): ")
    outlier_cloud.paint_uniform_color([1, 0, 0])
    inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries([inlier_cloud, outlier_cloud], window_name="Inliers (grey) and outliers (red)")
    if visualize_separately:
        o3d.visualization.draw_geometries([inlier_cloud], window_name="Inlier cloud")
        o3d.visualization.draw_geometries([outlier_cloud], window_name="Outlier cloud")

display_inlier_outlier(pcd.to_legacy(), clean_idx, visualize_separately=False)

In [ ]:
#o3d.io.write_point_cloud("removed_points.ply", pcd.to_legacy().select_by_index(clean_idx, invert=True))

## Quick test classical meshing methods

In [ ]:
# Load and visualize a sample leaf point cloud
#sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/19_maize_control_plant3_D00_leaf1_dense.ply")
sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense.ply")
#sample_leaf_path = Path("../data/TrackPlant3D/examples/single_leafs/maize/1_maize_control_plant1_D00_leaf1_dense_downsampled-voxel02.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_np = np.asarray(leaf_pcd.points)
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")

# Get normals too 
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])

o3d.visualization.draw_geometries([leaf_pcd])

### Apha shapes

In [ ]:
mesh_alpha = classical_methods.alpha_shape_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_alpha]) # -> Not bad but has holes
mesh_alpha.get_surface_area()

### Bezier

In [ ]:
mesh_bezier = classical_methods.bezier_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bezier]) # -> Has some issues, needs tuning
mesh_bezier.get_surface_area()

### B-spline

In [ ]:
mesh_bspline = classical_methods.bspline_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bspline]) # -> Looks better than bezier, but has some singularities
mesh_bspline.get_surface_area()

### Delaunay

In [ ]:
mesh_delaunay = classical_methods.delaunay_mesh_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_delaunay])
mesh_delaunay.get_surface_area()

### Poisson

In [ ]:
mesh_poisson = classical_methods.poisson_reconstruction(leaf_np, depth=9)
o3d.visualization.draw_geometries([mesh_poisson])
mesh_poisson.get_surface_area()

### Ball pivoting

In [ ]:
mesh_bpa = classical_methods.ball_pivoting_reconstruction(leaf_np, radius_multiplier=5.0)
o3d.visualization.draw_geometries([mesh_bpa])
mesh_bpa.get_surface_area()

Alternatively the area could be computed as below but it gives the same results

In [ ]:
vertices_bezier = np.asarray(mesh_bezier.vertices)
triangles_bezier = np.asarray(mesh_bezier.triangles)

area_bezier_manual = classical_methods.compute_mesh_area(vertices_bezier, triangles_bezier)
float(area_bezier_manual)

## Trimmed Poisson

In [ ]:
# Load and visualize a sample leaf point cloud
# sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/11_maize_control_plant2_D01_leaf1_dense.ply")
# leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))

crop = 'maize' # 'tomato'
idx = 53
leaf_paths = natsorted(Path(f"../data/TrackPlant3D/dense_leaves/{crop}").iterdir())
leaf_pcd = o3d.io.read_point_cloud(str(leaf_paths[idx]))
leaf_np = np.asarray(leaf_pcd.points)

# leaf_pcd.estimate_normals()
# leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
# leaf_pcd.orient_normals_to_align_with_direction()
# point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
# print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
# o3d.visualization.draw_geometries([leaf_pcd])

### Reconstruct Poisson surface

In [ ]:
#pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

With Meshlab

In [ ]:
# ms = pymeshlab.MeshSet()
# mesh = pymeshlab.Mesh(vertex_matrix=leaf_np)
# ms.add_mesh(mesh, "leaf")
# ms.compute_normal_for_point_clouds(k=20, smoothiter=10)
# ms.generate_surface_reconstruction_screened_poisson(depth=7)
# mesh = ms.current_mesh()

# vertices = np.asarray(mesh.vertex_matrix())
# triangles = np.asarray(mesh.face_matrix())

# o3d_mesh = o3d.geometry.TriangleMesh()
# o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
# o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
# o3d_mesh.compute_vertex_normals() # this makes faces look good
# o3d.visualization.draw_geometries([o3d_mesh])

Try poisson with open3d

In [ ]:
pcd = o3d.t.io.read_point_cloud(str(leaf_paths[idx]))
pcd.estimate_normals()
pcd.orient_normals_consistent_tangent_plane(k=50)  # This is very important!!
pcd.orient_normals_to_align_with_direction() # Putting this after consistent_tangent_plane seems to work better for normals to point outward from the inside face of the leaf
o3d.visualization.draw_geometries([pcd.to_legacy()], point_show_normal=True)

In [ ]:
# pcd = o3d.t.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(leaf_np)

# # Estimate normals
# pcd.estimate_normals()
# pcd.orient_normals_to_align_with_direction()
# pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd.to_legacy(), depth=7, scale=1.1)
mesh.compute_vertex_normals()
mesh.orient_triangles()
#mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))

# Plot based on density
densities = np.asarray(densities)
density_colors = plt.cm.plasma(densities / np.max(densities))[:, :3]  # Use colormap
mesh.vertex_colors = o3d.utility.Vector3dVector(density_colors)
o3d.visualization.draw_geometries([mesh])

### Trimming based on density

In [ ]:
def apply_threshold_and_visualize(mesh, densities, threshold):
    """Apply threshold and show before/after comparison"""
    
    # Get original data
    vertices = np.asarray(mesh.vertices)
    triangles = np.asarray(mesh.triangles)
    
    # Create mask for vertices to keep
    vertices_to_keep = densities >= threshold
    
    # Create mapping from old indices to new indices
    vertex_mapping = np.full(len(vertices), -1)
    vertex_mapping[vertices_to_keep] = np.arange(np.sum(vertices_to_keep))
    
    # Filter vertices
    new_vertices = vertices[vertices_to_keep]
    
    # Filter triangles - keep only triangles where all vertices are kept
    triangle_mask = np.all(vertices_to_keep[triangles], axis=1)
    new_triangles = triangles[triangle_mask]
    
    # Remap triangle indices
    new_triangles = vertex_mapping[new_triangles]
    
    # Create new mesh
    trimmed_mesh = o3d.geometry.TriangleMesh()
    trimmed_mesh.vertices = o3d.utility.Vector3dVector(new_vertices)
    trimmed_mesh.triangles = o3d.utility.Vector3iVector(new_triangles)
    trimmed_mesh.compute_vertex_normals()

    
    # Copy colors if they exist
    if mesh.has_vertex_colors():
        vertex_colors = np.asarray(mesh.vertex_colors)
        new_vertex_colors = vertex_colors[vertices_to_keep]
        trimmed_mesh.vertex_colors = o3d.utility.Vector3dVector(new_vertex_colors)
    
    # Show statistics
    original_vertices = len(vertices)
    remaining_vertices = len(new_vertices)
    print(f"Original vertices: {original_vertices}")
    print(f"Remaining vertices: {remaining_vertices}")
    print(f"Removed: {original_vertices - remaining_vertices} ({100*(1-remaining_vertices/original_vertices):.1f}%)")
    
    # Visualize both
    print("Original mesh (press Q to close and see trimmed)")
    o3d.visualization.draw_geometries([mesh], window_name="Original Mesh")
    
    print("Trimmed mesh")
    o3d.visualization.draw_geometries([trimmed_mesh], window_name="Trimmed Mesh")
    
    return trimmed_mesh

In [ ]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(leaf_np)

# Estimate normals
pcd.estimate_normals()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!
pcd.orient_normals_to_align_with_direction()

# Create mesh using Poisson Surface Reconstruction
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=7, scale=1.1)
mesh.compute_vertex_normals()
mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))
o3d.visualization.draw_geometries([mesh])

In [ ]:
selected_threshold = np.percentile(densities, 25)  # Adjust based on your plots
trimmed_mesh = apply_threshold_and_visualize(mesh, densities, selected_threshold)
o3d.visualization.draw_geometries([pcd, trimmed_mesh], window_name="Trimmed Mesh")
# o3d.io.write_triangle_mesh("trimmed_leaf_mesh.ply", trimmed_mesh)
# pcd, mesh = mesh_leaf(leaf_np, method="poisson", backend="pymeshlab")

### Trimming based on density + distance

Option 1: based on alpha shape boundary

In [ ]:
def create_alpha_shape_boundary(pcd, alpha=0.1):
    """Create alpha shape to define better boundaries"""
    
    # Create alpha shape (2D or 3D)
    alpha_shape = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    alpha_shape.compute_vertex_normals()
    
    return alpha_shape

def combine_poisson_with_alpha_boundary(pcd, poisson_depth=7, alpha=0.1, density_threshold_percentile=10):
    """Combine Poisson reconstruction with alpha shape boundary"""
    
    # Step 1: Poisson reconstruction
    poisson_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=poisson_depth
    )
    
    # Step 2: Create alpha shape for boundary reference
    alpha_shape = create_alpha_shape_boundary(pcd, alpha)
    
    # Step 3: Filter Poisson mesh by density
    density_threshold = np.percentile(densities, density_threshold_percentile)
    density_mask = densities >= density_threshold

    # Step 4: Additional filtering - keep vertices close to alpha shape
    poisson_vertices = np.asarray(poisson_mesh.vertices)
    alpha_vertices = np.asarray(alpha_shape.vertices)
    
    # Find distances from Poisson vertices to alpha shape vertices
    from scipy.spatial.distance import cdist
    distances = cdist(poisson_vertices, alpha_vertices)
    min_distances = np.min(distances, axis=1)
    
    # Additional mask: keep vertices close to alpha shape
    distance_threshold = np.percentile(min_distances, 80)  # Keep 80% closest
    distance_mask = min_distances <= distance_threshold
    
    # Step 5: Combine filters and remove vertices
    vertices_to_remove = ~(density_mask & distance_mask)
    print(f"Removing {np.sum(vertices_to_remove)}/{len(vertices_to_remove)} vertices")
    
    # Step 6: Apply filtering
    poisson_mesh.remove_vertices_by_mask(vertices_to_remove)
    poisson_mesh.compute_vertex_normals()
    poisson_mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))

    return poisson_mesh

In [ ]:
# Check out alpha shape
trimmed_mesh = combine_poisson_with_alpha_boundary(leaf_pcd, alpha=0.6)

print("Original points")
o3d.visualization.draw_geometries([leaf_pcd])

print("Trimmed mesh (density + distance filtering)")
o3d.visualization.draw_geometries([trimmed_mesh])

# Optional: visualize with point cloud overlay
print("Improved mesh with original points")
o3d.visualization.draw_geometries([leaf_pcd, trimmed_mesh])

In [ ]:
trimmed_mesh.get_surface_area()

Option 2: based on just distance from the point cloud

In [ ]:
def improve_mesh_boundary_simple(pcd, poisson_depth=9, density_percentile=15, distance_multiplier=2.0):
    """
    Improve Poisson mesh boundary by filtering vertices that are too far from original points
    """
    
    # Step 1: Poisson reconstruction
    print("Creating Poisson mesh...")
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, depth=poisson_depth
    )
    
    # Step 2: Filter by density
    density_threshold = np.percentile(densities, density_percentile)
    density_mask = densities >= density_threshold
    print(f"Density filtering keeps: {np.sum(density_mask)}/{len(density_mask)} vertices")
    
    # Step 3: Calculate distance threshold based on point cloud spacing
    pcd_points = np.asarray(pcd.points)
    from sklearn.neighbors import NearestNeighbors
    
    # Find average spacing in point cloud
    nbrs = NearestNeighbors(n_neighbors=5, algorithm='kd_tree').fit(pcd_points)
    distances_pcd, _ = nbrs.kneighbors(pcd_points)
    avg_spacing = np.mean(distances_pcd[:, 1])
    distance_threshold = distance_multiplier * avg_spacing
    
    print(f"Average point cloud spacing: {avg_spacing:.6f}")
    print(f"Using distance threshold: {distance_threshold:.6f}")
    
    # Step 4: Filter mesh vertices by distance to original points
    mesh_vertices = np.asarray(mesh.vertices)
    nbrs_mesh = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(pcd_points)
    distances_to_pcd, _ = nbrs_mesh.kneighbors(mesh_vertices)
    distances_to_pcd = distances_to_pcd.flatten()
    
    distance_mask = distances_to_pcd <= distance_threshold
    print(f"Distance filtering keeps: {np.sum(distance_mask)}/{len(distance_mask)} vertices")
    
    # Step 5: Combine filters and remove vertices
    vertices_to_remove = ~(density_mask & distance_mask)
    print(f"Removing {np.sum(vertices_to_remove)}/{len(vertices_to_remove)} vertices")
    
    # Step 6: Apply filtering
    mesh.remove_vertices_by_mask(vertices_to_remove)
    mesh.compute_vertex_normals()
    mesh.vertex_colors = o3d.utility.Vector3dVector(np.empty((0, 3)))

    return mesh, distances_to_pcd

In [ ]:
trimmed_mesh, distances = improve_mesh_boundary_simple(
    leaf_pcd, 
    poisson_depth=7,
    density_percentile=15,
    distance_multiplier=10  # Adjust this to make boundary tighter (smaller) or looser (larger)
)

print("Original points")
o3d.visualization.draw_geometries([leaf_pcd])

print("Trimmed mesh (density + distance filtering)")
o3d.visualization.draw_geometries([trimmed_mesh])

# Optional: visualize with point cloud overlay
print("Trimmed mesh with original points")
o3d.visualization.draw_geometries([leaf_pcd, trimmed_mesh])

In [ ]:
trimmed_mesh.get_surface_area() # This seems to give the most reasonable visual results and plausible area

In [ ]:
#success = o3d.io.write_triangle_mesh("trimmed_mesh.ply", trimmed_mesh)

### Trimming based on density + distance + boundary (WIP)

In [ ]:
# idx = 17
# leaf_paths = natsorted(Path("../data/TrackPlant3D/dense_leaves/maize").iterdir())
# leaf_paths[idx]

In [ ]:
#pcd = o3d.t.io.read_point_cloud(str(leaf_paths[idx]))
pcd = o3d.t.geometry.PointCloud(o3d.core.Tensor(leaf_np))
pcd.estimate_normals()
pcd.orient_normals_to_align_with_direction()
pcd.orient_normals_consistent_tangent_plane(k=30)  # This is very important!!

In [ ]:
# Boundary detection (angle of neighbours within a radius, requires normals, and API only exists in tensor-based pcd)
# https://www.open3d.org/docs/latest/python_api/open3d.t.geometry.PointCloud.html#open3d.t.geometry.PointCloud.compute_boundary_points
edge = pcd.compute_boundary_points(radius=10, max_nn=30, angle_threshold=180)
o3d.visualization.draw_geometries([edge[0].to_legacy()])

### Smooth mesh edges

In [ ]:
o3d.visualization.draw_geometries([trimmed_mesh])

In [ ]:
trimmed_mesh.compute_adjacency_list()
triangles = np.asarray(trimmed_mesh.triangles)
triangles

In [ ]:
# Count how many times each edge appears
edge_count = defaultdict(int)
for tri in triangles:
    for i in range(3):
        v1, v2 = sorted([tri[i], tri[(i+1)%3]])
        edge_count[(int(v1), int(v2))] += 1

# Boundary edges = edges seen only once
boundary_edges = [edge for edge, count in edge_count.items() if count == 1]

# Collect unique boundary vertices
boundary_vertices = sorted(set([v for edge in boundary_edges for v in edge]))
len(boundary_vertices)

In [ ]:
# Boundary vertex coordinates
verts = np.asarray(trimmed_mesh.vertices)
boundary_points = verts[boundary_vertices]

# Create point cloud for visualization
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(boundary_points)
o3d.visualization.draw_geometries([pcd])

In [ ]:
def smooth_boundary(mesh, boundary_vertices, iterations=50, alpha=0.5):
    mesh_out = copy.deepcopy(mesh)
    verts = np.asarray(mesh_out.vertices)
    adjacency = mesh_out.adjacency_list

    for _ in range(iterations):
        print(f"Smoothing iteration {_+1}/{iterations}")
        for v in boundary_vertices:
            neighbors = list(adjacency[v])
            if len(neighbors) > 0:
                avg = np.mean(verts[neighbors], axis=0)
                verts[v] = (1 - alpha) * verts[v] + alpha * avg

    mesh_out.vertices = o3d.utility.Vector3dVector(verts)
    return mesh_out

trimmed_mesh.compute_adjacency_list()
smoothed = smooth_boundary(trimmed_mesh, boundary_vertices, iterations=200, alpha=0.3)

o3d.visualization.draw_geometries([trimmed_mesh], window_name="Before Smoothing")
o3d.visualization.draw_geometries([smoothed], window_name="After Smoothing")

In [ ]:
verts = np.asarray(trimmed_mesh.vertices)
boundary_points = verts[boundary_vertices]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(boundary_points)
pcd.paint_uniform_color([1, 0, 0])  # red points

o3d.visualization.draw_geometries([trimmed_mesh, pcd])

In [ ]:
def get_boundary_loops(mesh):
    import numpy as np
    from collections import defaultdict

    triangles = np.asarray(mesh.triangles)
    edge_count = defaultdict(int)

    # Count edges
    for tri in triangles:
        for i in range(3):
            v1, v2 = sorted([tri[i], tri[(i+1)%3]])
            edge_count[(v1, v2)] += 1

    # Extract boundary edges (appear only once)
    boundary_edges = [edge for edge, count in edge_count.items() if count == 1]

    # Build adjacency for boundary vertices
    adjacency = defaultdict(list)
    for v1, v2 in boundary_edges:
        adjacency[v1].append(v2)
        adjacency[v2].append(v1)

    # Traverse loops
    loops = []
    visited = set()

    for start in adjacency:
        if start in visited:
            continue
        loop = []
        current = start
        prev = None
        while True:
            loop.append(current)
            visited.add(current)

            # Pick the next neighbor (avoid going back to prev)
            neighbors = adjacency[current]
            nxt = [n for n in neighbors if n != prev]
            if not nxt:
                break
            nxt = nxt[0]
            prev, current = current, nxt

            if current == start:
                break
        loops.append(loop)

    return loops

In [ ]:
loops = get_boundary_loops(trimmed_mesh)
print(f"Found {len(loops)} boundary loops")
for i, loop in enumerate(loops):
    print(f"Loop {i}: {len(loop)} vertices")

In [ ]:
verts = np.asarray(trimmed_mesh.vertices)

for loop in loops:
    boundary_points = verts[loop]

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(boundary_points)
    pcd.paint_uniform_color([1, 0, 0])  # red for loop

    o3d.visualization.draw_geometries([trimmed_mesh, pcd])

In [ ]:
from scipy import interpolate

def smooth_and_resample_loop(loop_vertices, smoothing=0.01, num_points=200):
    """
    Fit a spline to boundary loop and resample evenly.
    
    Parameters
    ----------
    loop_vertices : (N,3) ndarray
        Coordinates of the loop vertices (ordered).
    smoothing : float
        Spline smoothing factor (higher = smoother).
    num_points : int
        Number of resampled points.
    """
    x, y, z = loop_vertices[:,0], loop_vertices[:,1], loop_vertices[:,2]

    # Parameterize curve by arc length
    t = np.linspace(0, 1, len(loop_vertices))
    
    # Fit parametric spline (x(t), y(t), z(t))
    tck, u = interpolate.splprep([x, y, z], s=smoothing, per=True)  # per=True closes the loop
    
    # Resample evenly
    u_fine = np.linspace(0, 1, num_points)
    x_new, y_new, z_new = interpolate.splev(u_fine, tck)
    
    return np.vstack([x_new, y_new, z_new]).T

In [ ]:
# Assume we already have loops = get_boundary_loops(mesh)
verts = np.asarray(trimmed_mesh.vertices)

smoothed_loops = []
for loop in loops:
    loop_points = verts[loop]
    smooth_points = smooth_and_resample_loop(loop_points, smoothing=0.001, num_points=300)
    smoothed_loops.append(smooth_points)

In [ ]:
pcd_list = []
for smooth_points in smoothed_loops:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(smooth_points)
    pcd.paint_uniform_color([1, 0, 0])  # red curve
    pcd_list.append(pcd)

o3d.visualization.draw_geometries([trimmed_mesh] + pcd_list)

In [ ]:
import numpy as np
import open3d as o3d
from scipy import interpolate
from collections import defaultdict

def get_boundary_loops(mesh):
    """Extract boundary loops from mesh."""
    triangles = np.asarray(mesh.triangles)
    edge_count = defaultdict(int)

    # Count edges
    for tri in triangles:
        for i in range(3):
            v1, v2 = sorted([tri[i], tri[(i+1)%3]])
            edge_count[(v1, v2)] += 1

    # Extract boundary edges (appear only once)
    boundary_edges = [edge for edge, count in edge_count.items() if count == 1]

    # Build adjacency for boundary vertices
    adjacency = defaultdict(list)
    for v1, v2 in boundary_edges:
        adjacency[v1].append(v2)
        adjacency[v2].append(v1)

    # Traverse loops
    loops = []
    visited = set()

    for start in adjacency:
        if start in visited:
            continue
        loop = []
        current = start
        prev = None
        
        while True:
            loop.append(current)
            visited.add(current)

            # Pick the next neighbor (avoid going back to prev)
            neighbors = adjacency[current]
            nxt = [n for n in neighbors if n != prev]
            if not nxt:
                break
            nxt = nxt[0]
            prev, current = current, nxt

            if current == start:
                break
        
        if len(loop) > 2:  # Only keep meaningful loops
            loops.append(loop)

    return loops

def smooth_and_resample_loop(loop_vertices, smoothing=0.01, num_points=200):
    """
    Fit a spline to boundary loop and resample evenly.
    
    Parameters
    ----------
    loop_vertices : (N,3) ndarray
        Coordinates of the loop vertices (ordered).
    smoothing : float
        Spline smoothing factor (0 = interpolation, higher = smoother).
    num_points : int
        Number of resampled points.
    """
    if len(loop_vertices) < 4:
        print("Warning: Loop has fewer than 4 points, skipping smoothing")
        return loop_vertices
    
    x, y, z = loop_vertices[:,0], loop_vertices[:,1], loop_vertices[:,2]
    
    # Calculate cumulative arc length for better parameterization
    diffs = np.diff(loop_vertices, axis=0)
    distances = np.sqrt(np.sum(diffs**2, axis=1))
    cumulative_distances = np.concatenate([[0], np.cumsum(distances)])
    # Close the loop
    total_length = cumulative_distances[-1] + np.linalg.norm(loop_vertices[0] - loop_vertices[-1])
    t = cumulative_distances / total_length
    
    try:
        # Fit parametric spline (x(t), y(t), z(t))
        # k=3 for cubic spline, per=True for periodic (closed) curve
        tck, u = interpolate.splprep([x, y, z], u=t, s=smoothing, per=True, k=min(3, len(x)-1))
        
        # Resample evenly
        u_fine = np.linspace(0, 1, num_points, endpoint=False)  # Don't duplicate start/end
        x_new, y_new, z_new = interpolate.splev(u_fine, tck)
        
        return np.column_stack([x_new, y_new, z_new])
    
    except Exception as e:
        print(f"Spline fitting failed: {e}")
        print("Falling back to simple smoothing...")
        return simple_smooth_loop(loop_vertices, num_points)

def simple_smooth_loop(loop_vertices, num_points=200, iterations=3):
    """Fallback: simple averaging smoothing + resampling."""
    smoothed = loop_vertices.copy()
    
    # Apply Laplacian smoothing
    for _ in range(iterations):
        new_smoothed = smoothed.copy()
        n = len(smoothed)
        for i in range(n):
            prev_idx = (i - 1) % n
            next_idx = (i + 1) % n
            new_smoothed[i] = 0.25 * smoothed[prev_idx] + 0.5 * smoothed[i] + 0.25 * smoothed[next_idx]
        smoothed = new_smoothed
    
    # Resample to desired number of points
    if num_points != len(smoothed):
        # Calculate cumulative distances for resampling
        diffs = np.diff(smoothed, axis=0)
        distances = np.sqrt(np.sum(diffs**2, axis=1))
        cumulative_distances = np.concatenate([[0], np.cumsum(distances)])
        total_length = cumulative_distances[-1] + np.linalg.norm(smoothed[0] - smoothed[-1])
        
        # Interpolate to get evenly spaced points
        target_distances = np.linspace(0, total_length, num_points, endpoint=False)
        resampled = np.zeros((num_points, 3))
        
        for i, target_dist in enumerate(target_distances):
            # Find the segment containing this distance
            seg_idx = np.searchsorted(cumulative_distances, target_dist) - 1
            seg_idx = max(0, min(seg_idx, len(smoothed) - 2))
            
            # Linear interpolation within the segment
            if seg_idx < len(cumulative_distances) - 1:
                t = (target_dist - cumulative_distances[seg_idx]) / distances[seg_idx]
                t = np.clip(t, 0, 1)
                resampled[i] = (1 - t) * smoothed[seg_idx] + t * smoothed[seg_idx + 1]
            else:
                resampled[i] = smoothed[-1]
        
        return resampled
    
    return smoothed

def update_mesh_boundary(mesh, loops, smoothed_loops):
    """
    Update mesh vertices with smoothed boundary points.
    
    Parameters
    ----------
    mesh : o3d.geometry.TriangleMesh
        Original mesh
    loops : list of lists
        Original boundary vertex indices
    smoothed_loops : list of ndarrays
        Smoothed boundary coordinates
    """
    vertices = np.asarray(mesh.vertices).copy()
    
    for loop_indices, smoothed_coords in zip(loops, smoothed_loops):
        if len(loop_indices) == len(smoothed_coords):
            # Direct replacement
            vertices[loop_indices] = smoothed_coords
        else:
            # Need to handle resampling case - this is more complex
            # For now, just update existing vertices with closest smoothed points
            for i, vertex_idx in enumerate(loop_indices):
                if i < len(smoothed_coords):
                    vertices[vertex_idx] = smoothed_coords[i]
    
    # Update mesh
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    mesh.compute_vertex_normals()
    return mesh

# Usage example:
def smooth_mesh_boundary(mesh, smoothing=0.01, num_points=None, visualize=True):
    """Complete workflow to smooth mesh boundary."""
    
    # Extract boundary loops
    loops = get_boundary_loops(mesh)
    print(f"Found {len(loops)} boundary loops")
    
    # Get vertex coordinates
    vertices = np.asarray(mesh.vertices)
    
    smoothed_loops = []
    for i, loop in enumerate(loops):
        loop_vertices = vertices[loop]
        print(f"Loop {i}: {len(loop)} vertices")
        
        # Use original number of points if not specified
        target_points = num_points if num_points else len(loop)
        
        # Smooth and resample
        smoothed = smooth_and_resample_loop(loop_vertices, smoothing, target_points)
        smoothed_loops.append(smoothed)
        
        if visualize:
            # Visualize original vs smoothed
            original_pcd = o3d.geometry.PointCloud()
            original_pcd.points = o3d.utility.Vector3dVector(loop_vertices)
            original_pcd.paint_uniform_color([1, 0, 0])  # Red for original
            
            smoothed_pcd = o3d.geometry.PointCloud()
            smoothed_pcd.points = o3d.utility.Vector3dVector(smoothed)
            smoothed_pcd.paint_uniform_color([0, 1, 0])  # Green for smoothed
            
            o3d.visualization.draw_geometries([mesh, original_pcd, smoothed_pcd])
    
    # Update mesh with smoothed boundaries
    smoothed_mesh = update_mesh_boundary(mesh, loops, smoothed_loops)
    
    return smoothed_mesh, loops, smoothed_loops


In [ ]:
smoothed_mesh = smooth_mesh_boundary(trimmed_mesh, smoothing=0.1, num_points=100)

In [ ]:
o3d.visualization.draw_geometries([trimmed_mesh])
trimmed_mesh.filter_smooth_taubin(number_of_iterations=200, mu=0.5, lambda_=0.5)